## **Creando la Dimensión "Time"**

##### Paso 1 - Leer la tabla de hechos "fact_device" usando "spark.sql"

In [26]:
time_df = spark.sql("select * from lh_gold.dbo.fact_device")

StatementMeta(, 32f6ad7b-377f-4c18-927c-7da815a1ed54, 35, Finished, Available, Finished, False)

##### Paso 2 - Seleccionar la columna "released_announces"

In [27]:
from pyspark.sql.functions import col
time_selected_df = time_df.select(col("released_announced"))

StatementMeta(, 32f6ad7b-377f-4c18-927c-7da815a1ed54, 36, Finished, Available, Finished, False)

In [28]:
display(time_selected_df)

StatementMeta(, 32f6ad7b-377f-4c18-927c-7da815a1ed54, 37, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 6d4e9f11-af9d-4c1c-9fe3-6a36312ea962)

##### Paso 3 - Eliminar Valores duplicados y hacer un "orden ascendente"

In [29]:
time_dup_and_order = time_selected_df.dropna(subset=["released_announced"]) \
    .dropDuplicates(subset=["released_announced"]) \
    .sort("released_announced", ascending=True)

StatementMeta(, 32f6ad7b-377f-4c18-927c-7da815a1ed54, 38, Finished, Available, Finished, False)

##### Paso 4 - Cambier el nombre de la columna "released_announced" a "date"

In [30]:
time_renamed = time_dup_and_order.withColumnRenamed("released_announced", "date")

StatementMeta(, 32f6ad7b-377f-4c18-927c-7da815a1ed54, 39, Finished, Available, Finished, False)

##### Paso 5 - Agregar columnas de "Time Intelligence" a la Dimension "Time"

In [31]:
from pyspark.sql.functions import year, month, dayofmonth, date_format, quarter, concat, lit

time_final_df = time_renamed.withColumn("year", year("date")) \
    .withColumn("month", month("date")) \
    .withColumn("day", dayofmonth("date")) \
    .withColumn("month_name", date_format("date", "MMMM")) \
    .withColumn("day_name", date_format("date", "EEEE")) \
    .withColumn("quarter", concat(lit("Q"), quarter("date")))


StatementMeta(, 32f6ad7b-377f-4c18-927c-7da815a1ed54, 40, Finished, Available, Finished, False)

##### Paso 6 - Escribir datos en el "lh_gold"

In [32]:
time_final_df.write.format("delta")\
.mode("overwrite")\
.saveAsTable("dim_time")

StatementMeta(, 32f6ad7b-377f-4c18-927c-7da815a1ed54, 41, Finished, Available, Finished, False)

In [33]:
%%sql
select * from dim_time limit 10

StatementMeta(, 32f6ad7b-377f-4c18-927c-7da815a1ed54, 42, Finished, Available, Finished, False)

<Spark SQL result set with 10 rows and 7 fields>